In [0]:
# Old file read
file_location = "/FileStore/tables/tips.csv"
raw_df = spark.read.csv(file_location, header=True, inferSchema=True)

In [0]:
# Clean Unity Catalog table read
raw_df = spark.table("workspace.default.tips")
display(raw_df.limit(5))

In [0]:
import mlflow
import mlflow.spark
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression

In [0]:
raw_df.printSchema()

In [0]:
raw_df.columns

In [0]:
train_df, test_df = raw_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train rows: {train_df.count()} | Test rows: {test_df.count()}")

In [0]:
categorical_cols = ["sex", "smoker", "day", "time"]
indexed_cols = [f"{col}_indexed" for col in categorical_cols]
numeric_cols = ["tip", "size"]

In [0]:
indexer = StringIndexer(
    inputCols=categorical_cols,
    outputCols=indexed_cols,
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=numeric_cols + indexed_cols,
    outputCol="features"
)

lr = LinearRegression(
    featuresCol="features",
    labelCol="total_bill",
    regParam=0.1,
    elasticNetParam=0.0
)

In [0]:
pipeline = Pipeline(stages=[indexer, assembler, lr])

In [0]:
with mlflow.start_run(run_name="pyspark_linear_regression_pipeline"):
    # Fit strictly on train data
    pipeline_model = pipeline.fit(train_df)
    
    # Predict on test data
    predictions = pipeline_model.transform(test_df)
    
    # Evaluate
    rmse_eval = RegressionEvaluator(labelCol="total_bill", predictionCol="prediction", metricName="rmse")
    mae_eval = RegressionEvaluator(labelCol="total_bill", predictionCol="prediction", metricName="mae")
    r2_eval = RegressionEvaluator(labelCol="total_bill", predictionCol="prediction", metricName="r2")
    
    rmse = rmse_eval.evaluate(predictions)
    mae = mae_eval.evaluate(predictions)
    r2 = r2_eval.evaluate(predictions)
    
    mlflow.log_params({"regParam": 0.1, "elasticNetParam": 0.0})
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    print(f"Results -> RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")

display(predictions.select("features", "total_bill", "prediction").limit(10))